In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("ONE-Lab/MixSet")

README.md:   0%|          | 0.00/4.07k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


Mixset_train.json:   0%|          | 0.00/5.23M [00:00<?, ?B/s]

Mixset_test.json:   0%|          | 0.00/1.01M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/600 [00:00<?, ? examples/s]

In [ ]:
print(ds['train'][1])  # Print the second sample (index 1) from the 'train' split of the Hugging Face Dataset

{'category': 'blog', 'id': 1, 'original': "Summer tv tv in the summer used to be a dead zone full of repeats or something. But now there are two great new shows on tv and i have been watching both religiously for the last week. Blow out the gayest concept ever that seemingly has zero gay people in sight. Hairstylist jonothan antin wants build a second location to his incredibly popular west hollywood salon but this time in beverly hills. Normally no one would care, certainly not i, but as i was flipping thru the channels i caught a glipse of jonothan, equally narcissistic and dull looking at a empty space on rodeo dr speak (in a public voice) 'i am going to build here the greatest salon in the history of the world ever!' and i was hooked. Not since rocco have i found such a moron compelling tv watching.", 'revised': 'Summer TV used to be a dead zone filled with reruns, but now there are two great new shows that I\'ve been watching religiously for the past week. One show features hairst

In [ ]:
# =====================================================================
# INSPECT DATASET STRUCTURE, ROW COUNT, AND COLUMN COUNT
# =====================================================================

import pandas as pd  # DataFrame library for tabular inspection

# Convert the dataset's 'train' split to a pandas DataFrame for easy structural inspection
df = ds['train'].to_pandas()  # Convert Hugging Face Dataset split into a pandas DataFrame

print("=" * 50)  # Top banner border
print("DATASET OVERVIEW:")  # Section header
print("=" * 50)  # Bottom banner border
print(f"- Rows: {df.shape[0]}")  # Print total number of rows
print(f"- Columns: {df.shape[1]}")  # Print total number of columns
print(f"- Column names: {list(df.columns)}")  # Print list of column names

print("\n" + "=" * 50)  # Top border for data types section
print("DATA TYPES:")  # Section header
print("=" * 50)  # Bottom border for data types section
print(df.dtypes)  # Print column data types

print("\n" + "=" * 50)  # Top border for preview section
print("FIRST 5 ROWS OF THE DATASET:")  # Section header
print("=" * 50)  # Bottom border for preview section
print(df.head())  # Display the first 5 rows of the DataFrame

📌 THÔNG TIN TỔNG QUAN BỘ DỮ LIỆU:
👉 Số dòng (Rows): 3000
👉 Số cột (Columns): 7
👉 Tên các cột: ['category', 'id', 'original', 'revised', 'mixset_category', 'binary', 'model']

📌 CẤU TRÚC DỮ LIỆU (DATA TYPES):
category           object
id                  int64
original           object
revised            object
mixset_category    object
binary             object
model              object
dtype: object

📌 5 DÒNG ĐẦU TIÊN TRONG BỘ DỮ LIỆU:
  category  id                                           original  \
0     blog   0  Another week just flew by me. I drove by the n...   
1     blog   1  Summer tv tv in the summer used to be a dead z...   
2     blog   2  What a week ... My right shoulder has tendonit...   
3     blog   3  Kindness of strangers... Attempted laundry tod...   
4     blog   4  So it is 11:14pm mst, and here i sit in my bed...   

                                             revised      mixset_category  \
0  Another week has flown by. I drove past the ne...  2llama_polish

In [ ]:
# Print unique values in the 'binary' column and the count of each class
print("Unique values in the 'binary' column:")  # Log unique label values
print(df['binary'].unique())  # Display distinct label values in 'binary'

print("\nLabel distribution counts:")  # Log class frequency summary
print(df['binary'].value_counts())  # Count occurrences of each unique label (including non-null values)

📌 Các giá trị unique trong cột 'binary':
['MGT' 'HWT']

📊 Thống kê số lượng từng nhãn:
binary
MGT    2500
HWT     500
Name: count, dtype: int64


In [ ]:
# =====================================================================
# FULL PIPELINE: MIXSET + ELECTRA + MAJORITY VOTING + BALANCED SVM
# =====================================================================

import torch  # PyTorch deep learning framework
import numpy as np  # Numerical operations and array handling
import pandas as pd  # Tabular data manipulation with DataFrames
import nltk  # Natural Language Toolkit for sentence splitting
nltk.download('punkt')  # Download punkt sentence tokenizer data
nltk.download('punkt_tab')  # Download punkt_tab tokenizer resources required by newer NLTK versions
from nltk.tokenize import sent_tokenize  # Split text into individual sentences
from transformers import AutoTokenizer, AutoModelForTokenClassification  # Hugging Face tokenizer and token-level classifier
from sklearn.model_selection import train_test_split  # Stratified train/test data split
from sklearn.svm import SVC  # Support Vector Classifier
from sklearn.metrics import classification_report, accuracy_score  # Classification evaluation metrics

# 1. Check compute device (GPU/CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Use CUDA GPU if available, otherwise CPU
print(f"Using device: {device}")  # Log active compute device

# =====================================================================
# STEP 1: LOAD AND PREPROCESS MIXSET DATASET
# =====================================================================
# Reuse the pandas DataFrame converted from `ds['train']` if available
try:  # Attempt to load dataset from memory or fallback CSV
    if 'ds' in globals():  # Check if Hugging Face dataset object `ds` exists in current scope
        df = ds['train'].to_pandas()  # Convert 'train' split to a pandas DataFrame
        print(f"Loaded data from Hugging Face Dataset with {len(df)} rows.")  # Print row count
    else:  # Fallback to local CSV file when `ds` is not defined
        df = pd.read_csv("mixset.csv")  # Load MixSet CSV file
        print(f"Loaded MixSet file with {len(df)} rows.")  # Print row count
except Exception as e:  # Handle loading errors gracefully
    print(f"Could not load dataset, using fallback/mock data if defined. Error: {e}")  # Log error message

# Normalize 'binary' labels: 'MGT' -> 1 (AI/Mixed), 'HWT' -> 0 (Human)
df['target'] = df['binary'].apply(lambda x: 1 if str(x).upper() == 'MGT' else 0)  # Map MGT to 1 and HWT to 0

# Inspect class distribution to monitor class imbalance
print("\nOriginal label distribution:")  # Log label counts header
print(df['target'].value_counts())  # Print count of each binary target class

# =====================================================================
# STEP 2: SENTENCE TOKENIZATION & SENTENCE-LEVEL LABEL PREPARATION
# =====================================================================
print("\nSplitting texts into sentences and preparing data...")  # Log sentence segmentation step
all_sentences = []  # Store extracted sentences across all rows
all_labels = []  # Store corresponding document/row target labels

for idx, row in df.iterrows():  # Iterate over dataset rows
    text = str(row['revised'])  # Extract revised text column as string
    label = row['target']  # Retrieve binary target label
    sentences = sent_tokenize(text)  # Segment text into sentences using NLTK
    for sent in sentences:  # Iterate over segmented sentences
        if len(sent.strip()) > 3:  # Filter out overly short fragments or whitespace
            all_sentences.append(sent)  # Keep valid sentence
            all_labels.append(label)  # Keep associated label

print(f"Total sentences extracted from dataset: {len(all_sentences)}")  # Print total sentence count

# =====================================================================
# STEP 3: INITIALIZE ELECTRA MODEL FOR TOKEN CLASSIFICATION
# =====================================================================
model_checkpoint = "google/electra-small-discriminator"  # Pretrained ELECTRA small discriminator checkpoint
print(f"\nLoading backbone model {model_checkpoint}...")  # Log model checkpoint download
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)  # Load matching tokenizer
model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=2)  # Load token classification model with 2 output classes
model.to(device)  # Move model weights to target device (GPU or CPU)
model.eval()  # Set model to evaluation mode (disables dropout)

# =====================================================================
# STEP 4: MAJORITY VOTING ALGORITHM (TOKEN -> SENTENCE BRIDGE)
# =====================================================================
def apply_majority_voting(sentence, model, tokenizer):
    """
    Majority Voting algorithm: Aggregates ELECTRA token-level predictions,
    filters valid class labels, and assigns the majority label to the sentence.
    """
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True, max_length=128).to(device)  # Tokenize sentence into PyTorch tensors on device
    with torch.no_grad():  # Disable gradient tracking during inference
        outputs = model(**inputs)  # Forward pass through token classification model
        predictions = torch.argmax(outputs.logits, dim=2)[0].cpu().numpy()  # Get predicted class ID per token position

    # Filter valid binary token predictions (0 or 1)
    valid_preds = [p for p in predictions if p in [0, 1]]  # Retain binary label predictions
    if not valid_preds:  # Fallback if no valid binary predictions are present
        return 0  # Default to class 0

    counts = {0: valid_preds.count(0), 1: valid_preds.count(1)}  # Count token votes for class 0 and class 1
    return max(counts, key=counts.get)  # Return class with the highest token vote count

print("Running Majority Voting to aggregate sentence labels...")  # Log majority voting execution
voted_labels = [apply_majority_voting(s, model, tokenizer) for s in all_sentences]  # Compute voted label for every sentence

# =====================================================================
# STEP 5: EXTRACT EMBEDDINGS & TRAIN CLASS-BALANCED SVM
# =====================================================================
print("Extracting sentence embeddings with ELECTRA...")  # Log embedding extraction start
def extract_sentence_embeddings(sentences, model, tokenizer):  # Compute sentence-level embeddings via mean pooling
    embeddings = []  # List to hold sentence embedding vectors
    with torch.no_grad():  # Disable gradient computation for inference speed and lower memory
        for sent in sentences:  # Encode each sentence
            inputs = tokenizer(sent, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)  # Tokenize and move tensors to device
            outputs = model.electra(**inputs)  # Run ELECTRA base encoder backbone
            # Mean-pool token hidden states to obtain a fixed-size sentence representation
            hidden_states = outputs.last_hidden_state.mean(dim=1).cpu().numpy()  # Average across sequence tokens and move to NumPy
            embeddings.append(hidden_states[0])  # Append 1D embedding vector
    return np.array(embeddings)  # Stack into a 2D NumPy array of shape (n_sentences, hidden_size)

X_embeddings = extract_sentence_embeddings(all_sentences, model, tokenizer)  # Extract feature matrix for all sentences
y_data = np.array(voted_labels)  # Convert voted labels to NumPy array

# Split into Train / Test (80% / 20%) with stratification when multiple classes are present
X_train, X_test, y_train, y_test = train_test_split(
    X_embeddings, y_data, test_size=0.2, random_state=42, stratify=y_data if len(np.unique(y_data)) > 1 else None  # Preserve class proportions when >=2 classes exist
)

print("Training SVM model (with class_weight='balanced')...")  # Log SVM training start
# Use class_weight='balanced' to adjust class penalties inversely to class frequencies
svm_classifier = SVC(kernel='linear', probability=True, class_weight='balanced')  # Linear SVM with probability estimates and automatic class weighting
svm_classifier.fit(X_train, y_train)  # Fit SVM on sentence embeddings and labels
print("Training complete!")  # Log completion

# =====================================================================
# STEP 6: EVALUATE PERFORMANCE ON TEST SET
# =====================================================================
y_pred = svm_classifier.predict(X_test)  # Predict test set labels
acc = accuracy_score(y_test, y_pred)  # Compute overall test accuracy

print("\n" + "=" * 50)  # Top banner border
print("MODEL EVALUATION RESULTS ON TEST SET:")  # Evaluation header
print("=" * 50)  # Bottom banner border
print(f"Accuracy: {acc * 100:.2f}%\n")  # Display test accuracy percentage
print("Classification Report:")  # Subheader for detailed metrics
print(classification_report(y_test, y_pred, target_names=['Human (HWT)', 'AI/Mixed (MGT)'], zero_division=0))  # Print precision, recall, and F1-score per class

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


🚀 Đang sử dụng thiết bị: cuda
✅ Đã tải thành công dữ liệu từ Hugging Face Dataset với 3000 dòng.

📊 Phân phối nhãn dữ liệu gốc:
target
1    2500
0     500
Name: count, dtype: int64

✂️ Đang tiến hành tách câu và chuẩn bị dữ liệu...
👉 Tổng số câu sau khi tách từ tập dữ liệu: 21394

⏳ Đang tải mô hình backbone google/electra-small-discriminator...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] ElectraForTokenClassification LOAD REPORT from: google/electra-small-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
classifier.weight                                 | MISSING    | 
classifier.bias                                   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🗳️ Đang chạy Majority Voting để tổng hợp nhãn...
🔄 Đang trích xuất embedding câu bằng ELECTRA...
🤖 Đang huấn luyện mô hình SVM (với class_weight='balanced')...
✅ Huấn luyện hoàn tất!

🎯 KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH TRÊN TẬP TEST:
👉 Accuracy: 94.95%

Báo cáo chi tiết (Classification Report):
                precision    recall  f1-score   support

   Human (HWT)       0.50      0.91      0.64       213
AI/Mixed (MGT)       1.00      0.95      0.97      4066

      accuracy                           0.95      4279
     macro avg       0.75      0.93      0.81      4279
  weighted avg       0.97      0.95      0.96      4279

